# 05 · First Stage 3 experiment: frozen V-JEPA 2.1-B -> continuous CAN

Goal: establish a clean external-CAN baseline before DACON class fine-tuning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
DATA_ROOT = DRIVE_ROOT / 'DATASET'
COMMA_ROOT = DATA_ROOT / 'comma2k19'
RAW_ROOT = COMMA_ROOT / 'raw'
PROCESSED_ROOT = COMMA_ROOT / 'processed' / 'v1'
MANIFEST_ROOT = DRIVE_ROOT / 'manifests' / 'stage3' / 'v1'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'stage3'
PRETRAINED_ROOT = DRIVE_ROOT / 'pretrained'

# Clone your repository if this runtime does not have it yet.
if not REPO.exists():
    raise RuntimeError('Clone Blackbox-Detection to /content/Blackbox-Detection first, then rerun this cell.')
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_ROOT      :', RAW_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)
# Install this repository through its existing pyproject.toml without replacing
# Colab's binary stack. Dependency versions in pyproject.toml are aligned to the
# DACON evaluation-server package list.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)


In [ ]:
import json, yaml, torch, pandas as pd
from torch.utils.data import DataLoader
from blackbox_detection.stage3.dataset import Stage3CANDataset
from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.models import VJEPA21DenseCAN
from blackbox_detection.stage3.trainer import CANTrainer

CFG_PATH = REPO / 'configs/stage3/vjepa21b_can.yaml'
cfg = yaml.safe_load(CFG_PATH.read_text())
stats = json.loads((MANIFEST_ROOT / 'target_stats.json').read_text())
VJEPA_REPO = Path('/content/vjepa2')
VJEPA_COMMIT = '45d025f636dfc58fc2426905fc4a1ab755b1c3e5'  # V-JEPA 2.1 release commit; pin for reproducibility
if not VJEPA_REPO.exists():
    !git clone -q https://github.com/facebookresearch/vjepa2.git /content/vjepa2
!git -C /content/vjepa2 checkout -q $VJEPA_COMMIT
print('V-JEPA commit:', subprocess.check_output(['git','-C',str(VJEPA_REPO),'rev-parse','HEAD'], text=True).strip())
VJEPA_CKPT = PRETRAINED_ROOT / 'vjepa2_1_vitb_dist_vitG_384.pt'
if not VJEPA_CKPT.exists():
    !wget -q -O "$VJEPA_CKPT" https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt

In [ ]:
torch.manual_seed(cfg['seed'])

dc = cfg['data']
train_ds = Stage3CANDataset(
    MANIFEST_ROOT / 'comma_train.csv', PROCESSED_ROOT,
    target_stats=stats, clip_len=dc['clip_len'], window_stride=dc['window_stride'],
    input_size=(dc['input_height'], dc['input_width']), random_flip=dc['train_random_flip'],
    max_windows=dc['max_train_windows'], seed=cfg['seed'],
)
val_ds = Stage3CANDataset(
    MANIFEST_ROOT / 'comma_val_id.csv', PROCESSED_ROOT,
    target_stats=stats, clip_len=dc['clip_len'], window_stride=dc['window_stride'],
    input_size=(dc['input_height'], dc['input_width']), random_flip=False,
    max_windows=dc['max_val_windows'], seed=cfg['seed']+1,
)

train_loader = DataLoader(train_ds, batch_size=cfg['training']['batch_size'], shuffle=True, num_workers=dc['num_workers'], pin_memory=True, persistent_workers=dc['num_workers']>0)
val_loader = DataLoader(val_ds, batch_size=cfg['training']['batch_size'], shuffle=False, num_workers=dc['num_workers'], pin_memory=True, persistent_workers=dc['num_workers']>0)
print('windows:', len(train_ds), len(val_ds))

In [ ]:
mc = cfg['model']
backbone = load_vjepa21_base_encoder(VJEPA_REPO, VJEPA_CKPT, num_frames=dc['clip_len'], out_layers=tuple(mc['out_layers']), freeze=mc['freeze_backbone'])
model = VJEPA21DenseCAN(
    backbone,
    freeze_backbone=mc['freeze_backbone'],
    feature_dim=mc['feature_dim'],
    temporal_hidden=mc['temporal_hidden'],
    temporal_layers=mc['temporal_layers'],
)
trainable = [p for p in model.parameters() if p.requires_grad]
print('trainable params:', sum(p.numel() for p in trainable)/1e6, 'M')
optimizer = torch.optim.AdamW(trainable, lr=cfg['training']['learning_rate'], weight_decay=cfg['training']['weight_decay'])

run_dir = OUTPUT_ROOT / 'vjepa21b_can_v1'
trainer = CANTrainer(
    model, optimizer,
    grad_accum_steps=cfg['training']['grad_accum_steps'],
    grad_clip_norm=cfg['training']['grad_clip_norm'],
    amp_dtype=cfg['training']['amp_dtype'],
    loss_weights=cfg['loss'], stats=stats, output_dir=run_dir,
)

In [ ]:
resume_path = run_dir / 'latest.pt'
print('resume:', resume_path if resume_path.exists() else None)

history = trainer.fit(
    train_loader, val_loader,
    epochs=cfg['training']['epochs'],
    max_train_steps=cfg['training']['max_steps_per_epoch'],
    max_val_steps=500,
    resume_from=resume_path if resume_path.exists() else None,
)
pd.DataFrame([{'epoch':x['epoch'], 'minutes':x['minutes'], **{f'train/{k}':v for k,v in x['train'].items()}, **{f'val/{k}':v for k,v in x['val'].items()}} for x in history])

Do not tune DACON class thresholds yet. First record this run's ID validation MAEs and training time. The next controlled ablations are: (1) more comma2k19 windows, (2) partial backbone unfreezing/LoRA, then (3) A2D2 and target-domain DACON heads.